# Homework 3A: Classification with Python

| Exercise | Topic | Points |
|---|---|---|
| 1 | Income classification (Decision Tree, KNN, Naïve Bayes) | 40 |
| 2 | Heart disease detection | 30 |
| 3 | Breast cancer detection | Extra 10 |

**Instructions**
- All data files are in the `Data/` folder next to this notebook.
- Use `random_state=42` wherever a function accepts it, so your results are reproducible.
- Write code in the code cells and answer the discussion questions in the markdown cells marked **Your answer**.
- Before submitting, choose **Kernel → Restart & Run All** and make sure every cell runs without errors. Submit the `.ipynb` file with all outputs visible.
- Allowed libraries: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`.

---
## Exercise 1: Income Classification (40 points)

`Data/cleaned_transformed_NewYork.csv` contains **45,171** census records of adults (derived from the UCI *Adult* census income dataset). Each record describes a person's demographics and work, such as age, education level, occupation, and hours worked per week. The categorical attributes have already been integer-encoded.

**Goal:** classify each person's income into one of two categories: `<=50K` or `>50K`.

> **Data notes (read before you start)**
> - The label appears **twice**: `income` (text) and `income_encoded` (0 = `<=50K`, 1 = `>50K`). Use `income_encoded` as the label and remove **both** columns from the features. If `income_encoded` is left in the features, the model can simply copy the answer.
> - `ID` is a row identifier, not a feature. Remove it as well.
> - The classes are imbalanced (about 75% `<=50K`).

In [ ]:
# 1.1 Import packages and load Data/cleaned_transformed_NewYork.csv as a pandas DataFrame
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB

SAVE_PATH = "pathway"

data = pd.read_csv(SAVE_PATH + "cleaned_transformed_NewYork.csv")

In [ ]:
# 1.2 Explore the data: shape, data types, missing values, summary statistics,
#     and the number of records in each income class

print("Shape:", data.shape, "\n\n")
print(data.dtypes, "\n\n")
print(data.isnull().sum(), "\n\n")
print(data["income_encoded"].value_counts(), "\n\n")

data.describe()

In [ ]:
# 1.3 Create the label y from income_encoded, and the feature matrix X by dropping
#     income, income_encoded, and ID

X = data.drop(columns=['income', 'income_encoded', 'ID'])
y = data["income_encoded"]

print(X.shape, y.shape, "\n\n")
print(X.columns.tolist())

In [ ]:
# 1.4 Split the data into a training set (80%) and a test set (20%)
#     Hint: train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape, "\n\n")

print(y_train.mean())
print(y_test.mean())

In [ ]:
# 1.5 Normalize the features. Choose a suitable method (e.g., min-max or z-score)
#     based on what you observed in 1.2.
#     Fit the scaler on the TRAINING set only, then transform both the training and test sets.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = X_train.copy()
X_test = X_test.copy()
for col in ["capital-gain", "capital-loss"]:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

print(X_train_scaled.mean(axis=0).round(2))
print(X_test_scaled.mean(axis=0).round(2), "\n\n")

In [ ]:
# 1.6 Decision Tree: train on the training set, predict the test set,
#     and report Precision / Recall / F1-score (e.g., with classification_report)

tree_raw = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_scaled = DecisionTreeClassifier(random_state=42).fit(X_train_scaled, y_train)

print(classification_report(y_test, tree_raw.predict(X_test)))
print(classification_report(y_test, tree_scaled.predict(X_test_scaled)))

In [ ]:
# 1.7 KNN: train on the training set, predict the test set,
#     and report Precision / Recall / F1-score

for k in [5, 15, 25]:
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_scaled, y_train, cv=5, scoring="f1")
    print(k, scores.mean().round(3))

In [ ]:
# 1.8 Naive Bayes: train on the training set, predict the test set,
#     and report Precision / Recall / F1-score

nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
y_pred_nb = nb.predict(X_test_scaled)

print(classification_report(y_test, y_pred_nb))

In [ ]:
# 1.9 Put the three models' Precision / Recall / F1-score in one table for comparison



**1.10 Compare the models and explain your observations.** Which model performs best, and why might that be? Your best model should reach a **weighted-average F1-score above 0.80** on the test set. Also comment on the F1-score of the minority class (`>50K`): what does it tell you that the weighted average hides?

**Your answer:**



---
## Exercise 2: Heart Disease Detection (30 points)

`Data/heart disease.csv` contains patient attributes for predicting the presence of heart disease (`target`: 1 = heart disease, 0 = no heart disease).

| Type | Columns |
|---|---|
| Continuous | `age`, `trestbps` (resting blood pressure), `chol` (cholesterol), `thalach` (max heart rate), `oldpeak` |
| Categorical (already integer-coded) | `sex`, `cp` (chest pain type), `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal` |

> **Data note:** the file has 1,025 rows, but only **302** are unique patients; the other 723 rows are exact duplicates. Remove the duplicates **before** splitting the data. Otherwise copies of the same patient land in both the training and test sets, and the test score will be unrealistically high.

In [ ]:
# 2.1 Import packages and load Data/heart disease.csv
#     Check for duplicate rows and remove them


In [ ]:
# 2.2 Preprocessing: normalize the continuous features.
#     The categorical features are already integer-coded; decide whether to keep those codes
#     or one-hot encode the nominal ones (cp, restecg, slope, thal), and explain your choice in 2.6.
#     As in Exercise 1, split first and fit any scaler on the training set only.


In [ ]:
# 2.3 Explore the relationships between features (e.g., a correlation heatmap)
#     and between each feature and the target


In [ ]:
# 2.4 Check whether the task is balanced by counting the records in each target class


In [ ]:
# 2.5 Choose one of the models (Decision Tree, KNN, or Naive Bayes) to detect heart disease.
#     Report the confusion matrix and Precision / Recall / F1-score on the test set


**2.6 Discussion.** Is the task balanced? Which model did you choose and why? In heart disease detection, which error is more costly, a false negative or a false positive, and which metric should you watch most closely?

**Your answer:**



---
## Exercise 3: Breast Cancer Detection (Extra 10 points)

`Data/Breast Cancer.csv` (Breast Cancer Wisconsin Diagnostic dataset) contains 30 features computed from digitized images of breast-mass biopsies, such as the radius, texture, perimeter, and area of the cell nuclei. It has 569 instances, and the goal is to predict whether a tumor is malignant (`M`) or benign (`B`). It is widely used as a benchmark for medical diagnosis.

> **Data notes:** `id` is an identifier and `Unnamed: 32` is an empty column; drop both. Encode `diagnosis` as 1 = `M` (malignant), 0 = `B` (benign).

In [ ]:
# 3.1 Load Data/Breast Cancer.csv, drop id and Unnamed: 32, and encode diagnosis (M = 1, B = 0)


In [ ]:
# 3.2 Split the data (80/20, stratified), preprocess, and train at least two classifiers


In [ ]:
# 3.3 Evaluate the classifiers on the test set and report the best performance
#     (confusion matrix and Precision / Recall / F1-score)


**3.4 Discussion.** Which model performed best? Report its Recall for the malignant class and explain why that metric matters here.

**Your answer:**

